In [4]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict


project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from environment.my_gridworld import EnhancedGridWorldEnv
from model.a2c import A2C


In [5]:
def train_a2c(
    env,
    num_episodes: int = 1000,
    gamma: float = 0.99,
    lr: float = 1e-3,
    entropy_coef: float = 0.08,
    value_coef: float = 0.5,
    max_steps_per_episode: int = 300,
    seed: int = 42
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    model = A2C(obs_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    rewards_log = []
    total_loss_log = []
    policy_loss_log = []
    value_loss_log = []
    entropy_log = []

    for ep in range(num_episodes):
        state, _ = env.reset(seed=seed + ep)
        ep_reward = 0.0
        done = False
        step = 0

        log_probs, values, rewards, dones = [], [], [], []

        while not done and step < max_steps_per_episode:
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits, value = model(state_tensor)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value.squeeze())
            rewards.append(reward)
            dones.append(done)
            ep_reward += reward

            state = next_state
            step += 1

        if done:
            last_value = 0.0
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                _, value_next = model(state_tensor)
                last_value = value_next.item()

        returns = []
        R = last_value
        for r, d in zip(reversed(rewards), reversed(dones)):
            R = r + gamma * R * (1 - d)
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32)

        log_probs = torch.stack(log_probs)
        values = torch.stack(values)
        advantages = returns - values.detach()

        policy_loss = -(log_probs * advantages).mean()
        value_loss = F.mse_loss(values, returns)
        entropy = -(log_probs * torch.exp(log_probs)).mean()
        total_loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        rewards_log.append(ep_reward)
        total_loss_log.append(total_loss.item())
        policy_loss_log.append(policy_loss.item())
        value_loss_log.append(value_loss.item())
        entropy_log.append(entropy.item())

        if ep % 100 == 0:
            avg_r = np.mean(rewards_log[-100:])
            print(f"[{env.env_id}] Ep {ep:4d} | Reward: {ep_reward:6.2f} | Avg100: {avg_r:6.2f}")

    logs = {
        "rewards": rewards_log,
        "total_loss": total_loss_log,
        "policy_loss": policy_loss_log,
        "value_loss": value_loss_log,
        "entropy": entropy_log,
    }
    return model, logs


def record_trajectory(env, model, max_steps=100):
    state, _ = env.reset()
    positions = [env.agent_pos]
    for _ in range(max_steps):
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        logits, _ = model(state_tensor)
        probs = F.softmax(logits, dim=-1)
        action = torch.multinomial(probs, 1).item()  #sampling вместо argmax
        state, _, done, _, _ = env.step(action)
        positions.append(env.agent_pos)
        if done:
            break
    return positions


def collect_visit_map(env, model, n_episodes=30):
    visit_count = np.zeros((env.height, env.width), dtype=int)
    for _ in range(n_episodes):
        state, _ = env.reset()
        for _ in range(300):
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits, _ = model(state_tensor)
            probs = F.softmax(logits, dim=-1)
            action = torch.multinomial(probs, 1).item()  
            state, _, done, _, _ = env.step(action)
            r, c = env.agent_pos
            if 0 <= r < env.height and 0 <= c < env.width:
                visit_count[r, c] += 1
            if done:
                break
    return visit_count


def plot_training_logs(logs_dict: Dict[str, Dict], save_dir: str = "results_a2c"):
    os.makedirs(save_dir, exist_ok=True)

    env_ids = list(logs_dict.keys())
    metrics = ["rewards", "total_loss", "policy_loss", "value_loss", "entropy"]
    titles = {
        "rewards": "Средняя награда за эпизод",
        "total_loss": "Общий лосс",
        "policy_loss": "Policy Loss",
        "value_loss": "Value Loss",
        "entropy": "Энтропия"
    }

    for metric in metrics:
        plt.figure(figsize=(12, 6))
        for env_id in env_ids:
            data = logs_dict[env_id][metric]
            window = min(50, len(data))
            cumsum = np.cumsum(np.insert(data, 0, 0))
            smoothed = (cumsum[window:] - cumsum[:-window]) / window
            plt.plot(smoothed, label=env_id)
        plt.xlabel("Episode")
        plt.ylabel(titles[metric])
        plt.title(titles[metric])
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, f"{metric}.png"))
        plt.close()


def plot_trajectory_and_heatmap(env, model, env_id: str, save_dir: str = "results_a2c"):
    traj = record_trajectory(env, model, max_steps=100)
    grid = np.zeros((env.height, env.width))
    for r, c in traj:
        if 0 <= r < env.height and 0 <= c < env.width:
            grid[r, c] = 1

    plt.figure(figsize=(6, 6))
    sns.heatmap(grid, cmap="Blues", cbar=False, linewidths=0.5, linecolor='gray')
    plt.title(f"Траектория агента ({env_id})")
    plt.savefig(os.path.join(save_dir, f"trajectory_{env_id}.png"))
    plt.close()

    # Тепловая карта
    visits = collect_visit_map(env, model, n_episodes=30)
    plt.figure(figsize=(6, 6))
    sns.heatmap(visits, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
    plt.title(f"Тепловая карта посещений ({env_id})")
    plt.savefig(os.path.join(save_dir, f"heatmap_{env_id}.png"))
    plt.close()

In [7]:

env1 = EnhancedGridWorldEnv(
    height=5, width=5, n_colors=25,
    pos_goal=(4, 4), pos_agent=(0, 0),
    max_steps=100, use_penalties=True, env_id="Env1"
)

env2 = EnhancedGridWorldEnv(
    height=5, width=5, n_colors=5,
    pos_goal=(4, 4), pos_agent=(0, 0),
    max_steps=100, use_penalties=True, env_id="Env2", seed=42
)

np.random.seed(42)
obs_mask3 = np.random.rand(10, 10) < 0.10
obs_mask3[0, 0] = obs_mask3[9, 9] = False
env3 = EnhancedGridWorldEnv(
    height=10, width=10, n_colors=7, obstacle_mask=obs_mask3,
    pos_goal=(9, 9), pos_agent=(0, 0),
    max_steps=300, env_id="Env3", use_penalties=True, seed=42
)

np.random.seed(43)
obs_mask4 = np.random.rand(10, 10) < 0.10
obs_mask4[0, 0] = obs_mask4[9, 9] = False
env4 = EnhancedGridWorldEnv(
    height=10, width=10, n_colors=4, obstacle_mask=obs_mask4,
    pos_goal=(9, 9), pos_agent=(0, 0),
    max_steps=300, use_penalties=True, env_id="Env4", seed=42
)

envs = [env1, env2, env3, env4]
all_logs = {}
models = {}

for env in envs:
    print(f"\nОбучение на {env.env_id}...")
    if "Env1" in env.env_id or "Env2" in env.env_id:
        model, logs = train_a2c(env, num_episodes=500, max_steps_per_episode=100, seed=42)
    else:
        model, logs = train_a2c(env, num_episodes=800, max_steps_per_episode=300, seed=42)
    all_logs[env.env_id] = logs
    models[env.env_id] = model
    env.close()

plot_training_logs(all_logs)

for env in [env1, env2, env3, env4]:
    model = models[env.env_id]
    plot_trajectory_and_heatmap(env, model, env.env_id)

print("результаты сохранены в папке 'results_a2c'")



Обучение на Env1...
[Env1] Ep    0 | Reward:  -0.92 | Avg100:  -0.92
[Env1] Ep  100 | Reward:  -0.48 | Avg100:  -1.81
[Env1] Ep  200 | Reward:   0.70 | Avg100:  -0.21
[Env1] Ep  300 | Reward:   0.72 | Avg100:   0.60
[Env1] Ep  400 | Reward:   0.76 | Avg100:   0.79

Обучение на Env2...
[Env2] Ep    0 | Reward:  -6.58 | Avg100:  -6.58
[Env2] Ep  100 | Reward:  -1.36 | Avg100:  -1.36
[Env2] Ep  200 | Reward:   0.92 | Avg100:   0.54
[Env2] Ep  300 | Reward:   0.88 | Avg100:   0.76
[Env2] Ep  400 | Reward:   0.92 | Avg100:   0.75

Обучение на Env3...
[Env3] Ep    0 | Reward: -12.20 | Avg100: -12.20
[Env3] Ep  100 | Reward: -19.36 | Avg100: -19.03
[Env3] Ep  200 | Reward:  -4.26 | Avg100: -12.91
[Env3] Ep  300 | Reward:  -2.80 | Avg100:  -4.46
[Env3] Ep  400 | Reward:  -0.42 | Avg100:  -2.46
[Env3] Ep  500 | Reward:  -6.48 | Avg100:  -1.74
[Env3] Ep  600 | Reward:  -0.62 | Avg100:  -1.40
[Env3] Ep  700 | Reward:  -1.46 | Avg100:  -1.31

Обучение на Env4...
[Env4] Ep    0 | Reward: -19.52 | 